In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Datamart Gold: Cohort Retention

Este notebook construye la tabla de análisis de cohortes.

**Lógica:**
- Cada usuario pertenece a una **cohorte** definida por su mes de primera compra.
- Por cada mes posterior, medimos cuántos usuarios de esa cohorte volvieron a comprar.
- El resultado es la base perfecta para un **Retention Grid / Heatmap** en cualquier herramienta de BI.

In [ ]:
# 1. Lectura de tablas Silver
df_trans = spark.table('castor.silver.slv_fact_transactions')
df_users = spark.table('castor.silver.slv_dim_users')

In [ ]:
# 2. Calcular la cohorte de cada usuario: año-mes de su PRIMERA compra
first_purchase_df = df_trans.groupBy('id_usuario').agg(
    F.date_format(F.min('fecha_transaccion'), 'yyyy-MM').alias('cohort_month')
)

In [ ]:
# 3. Calcular en qué año-mes realizó cada transacción
df_trans_with_period = df_trans.withColumn(
    'transaction_month', F.date_format('fecha_transaccion', 'yyyy-MM')
)

In [ ]:
# 4. Join: unir cada transacción con la cohorte de su usuario
df_with_cohort = df_trans_with_period.join(first_purchase_df, on='id_usuario', how='left')

In [ ]:
# 5. Calcular el 'period_index': cuántos meses han pasado desde la cohorte hasta la transacción
# Usamos los meses como enteros para hacer la diferencia
df_with_cohort = df_with_cohort.withColumn(
    'cohort_month_ts', F.to_date(F.concat(F.col('cohort_month'), F.lit('-01')))
).withColumn(
    'transaction_month_ts', F.to_date(F.concat(F.col('transaction_month'), F.lit('-01')))
).withColumn(
    'period_index',
    (F.year('transaction_month_ts') - F.year('cohort_month_ts')) * 12 +
    (F.month('transaction_month_ts') - F.month('cohort_month_ts'))
)

In [ ]:
# 6. Agregar: contar usuarios únicos activos por cohorte y por periodo
df_cohort_activity = df_with_cohort.groupBy('cohort_month', 'period_index').agg(
    F.countDistinct('id_usuario').alias('active_users'),
    F.sum('monto').alias('revenue_cohort')
)

In [ ]:
# 7. Calcular el tamaño de cada cohorte (usuarios en period_index = 0)
cohort_size_df = df_cohort_activity.filter(F.col('period_index') == 0).select(
    'cohort_month', F.col('active_users').alias('cohort_size')
)

In [ ]:
# 8. Calcular la tasa de retención: active_users del periodo / cohort_size inicial
df_retention = df_cohort_activity.join(cohort_size_df, on='cohort_month', how='left')

df_retention = df_retention.withColumn(
    'retention_rate',
    F.round((F.col('active_users') / F.col('cohort_size')), 4)
).select(
    'cohort_month',
    'period_index',
    'cohort_size',
    'active_users',
    'retention_rate',
    'revenue_cohort'
).orderBy('cohort_month', 'period_index')

display(df_retention)

In [ ]:
# 9. Guardar en Gold
# Particionamos por cohort_month porque las queries típicas filtran por cohorte
df_retention.write.mode('overwrite').partitionBy('cohort_month').saveAsTable('castor.gold.gld_cohort_retention')

print(f'Tabla castor.gold.gld_cohort_retention guardada con éxito.')
print(f'Total cohortes: {df_retention.select("cohort_month").distinct().count()}')